# Train Phishing Detection Model on Google Colab
This notebook fine-tunes a `distilbert-base-uncased` model for phishing URL detection.

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn

In [ ]:
import torch
import evaluate
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [ ]:
# Load a public phishing dataset from Hugging Face
# We use 'pirocheto/phishing-url' which is natively in Parquet format
dataset = load_dataset('pirocheto/phishing-url')

if 'test' not in dataset:
    dataset = dataset['train'].train_test_split(test_size=0.2)

print(dataset)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    # Attempt some fallback to identify text content in unknown datasets
    text_column = 'text' if 'text' in examples else 'url'
    if 'text' not in examples and 'url' not in examples:
        text_column = list(examples.keys())[0]
    
    # Tokenize the URLs
    tokenized = tokenizer(examples[text_column], padding="max_length", truncation=True, max_length=128)
    
    # Standardize label column to 'labels' for the Trainer
    label_column = 'label' if 'label' in examples else 'status'
    if label_column not in examples:
        # Last resort: find any column containing integers and assume it might be a label if named suggestively
        for key in examples:
            if 'phish' in key.lower() or 'class' in key.lower():
                label_column = key
                break
                
    # Map labels to integers if they are not already (e.g., if 'phishing' vs 'legitimate')
    if isinstance(examples[label_column][0], str):
        labels = [1 if 'phish' in str(l).lower() or 'bad' in str(l).lower() else 0 for l in examples[label_column]]
        tokenized['labels'] = labels
    else:
        tokenized['labels'] = examples[label_column]
        
    return tokenized

tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments(
    output_dir="./phishing_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Save the final model and tokenizer
model.save_pretrained("./phishblocker_distilbert")
tokenizer.save_pretrained("./phishblocker_distilbert")

!zip -r phishblocker_distilbert.zip ./phishblocker_distilbert
print("Model saved and zipped! Download phishblocker_distilbert.zip to your local machine.")